In [1]:
import numpy as np
import pandas as pd
import cooler

import matplotlib.pyplot as plt
from tqdm import tqdm, trange

from itertools import combinations
import pybedtools

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

### 比较MATCHA和H2P预测得到的1Mb MCI之间的差异：
    1. 集合关系：Venn、Jaccard、overlap coefficient、top-k overlap
    2. 基因组结构：cis/trans、最小距离、平均距离、总跨度、anchor 间距
    3. Pairwise Hi-C 支持：0、1、2、3 条边，clique density，平均边权，最小边权
    4. 细胞核结构：A/B compartment、subcompartment、loop/TAD overlap
    5. 生物学特征：H3K27ac、H3K4me1、H3K4me3、ATAC-seq、CTCF、SMC3、基因数量、super-enhancer overlap

In [7]:
#### HiPore-C GM12878 1Mb Unobserved_MCI_O3: H2P vs MATCHA

Unobserved_MCI_O3 = np.load('/data/xujs/Project/HiC2PoreC/code/H2P/Analysis_Results/2024-05-23_Predict_New_MCI/HiPore-C_GM12878_1Mb.Unobserved_MCI_O3.npy')
h2p_O3_preds_score = np.load('/data/xujs/Project/HiC2PoreC/code/H2P/Analysis_Results/2024-05-23_Predict_New_MCI/HiPore-C_GM12878_1Mb.Unobserved_MCI_O3_pred_score.npy').reshape(-1)
matcha_O3_preds_score = np.load('/data/xujs/Project/HiC2PoreC/code/H2P/Analysis_Results/2024-05-31_Model_comparison/MATCHA/1Mb_Code/HiPore-C_GM12878_1Mb.MATCHA_Unobserved_MCI_O3_pred_score.npy').reshape(-1)

In [52]:
Unobserved_MCI_O3[(h2p_O3_preds_score >= 0.5) & (matcha_O3_preds_score >= 0.5)].shape

(105966, 3)

In [86]:
# ## MATCHA预测为否，而H2P预测为是
# Unobserved_MCI_O3_is_h2p_not_matcha = Unobserved_MCI_O3[(matcha_O3_preds_score < 0.5) & (h2p_O3_preds_score > 0.76) & (h2p_O3_preds_score < 0.95)]
# Unobserved_MCI_O3_is_h2p_not_matcha_score = h2p_O3_preds_score[(matcha_O3_preds_score < 0.5) & (h2p_O3_preds_score > 0.76) & (h2p_O3_preds_score < 0.95)]
# ## MATCHA预测为是，而H2P预测为否
# # Unobserved_MCI_O3_is_matcha_not_h2p = Unobserved_MCI_O3[(matcha_O3_preds_score >= 0.5) & (h2p_O3_preds_score >= 0.1) & (h2p_O3_preds_score < 0.5)]
# Unobserved_MCI_O3_is_matcha_not_h2p = Unobserved_MCI_O3[(matcha_O3_preds_score >= 0.5) & (h2p_O3_preds_score < 0.5)]
# Unobserved_MCI_O3_is_matcha_not_h2p_score = matcha_O3_preds_score[(matcha_O3_preds_score >= 0.5) & (h2p_O3_preds_score < 0.5)]

# np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/sgmci_nodes.npy', Unobserved_MCI_O3_is_h2p_not_matcha)
# np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/sgmci_scores.npy', Unobserved_MCI_O3_is_h2p_not_matcha_score)

# np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/matcha_nodes.npy', Unobserved_MCI_O3_is_matcha_not_h2p)
# np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/matcha_scores.npy', Unobserved_MCI_O3_is_matcha_not_h2p_score)

In [131]:
sgmci_predicted_MCI_O3 = Unobserved_MCI_O3[((h2p_O3_preds_score >= 0.5) & (matcha_O3_preds_score >= 0.5)) | ((matcha_O3_preds_score < 0.5) & (h2p_O3_preds_score > 0.76) & (h2p_O3_preds_score < 0.95))]
sgmci_predicted_MCI_O3_score = h2p_O3_preds_score[((h2p_O3_preds_score >= 0.5) & (matcha_O3_preds_score >= 0.5)) | ((matcha_O3_preds_score < 0.5) & (h2p_O3_preds_score > 0.76) & (h2p_O3_preds_score < 0.95))]

matcha_predicted_MCI_O3 = Unobserved_MCI_O3[((h2p_O3_preds_score >= 0.5) & (matcha_O3_preds_score >= 0.5)) | ((matcha_O3_preds_score >= 0.5) & (h2p_O3_preds_score < 0.5))]
matcha_predicted_MCI_O3_score = matcha_O3_preds_score[((h2p_O3_preds_score >= 0.5) & (matcha_O3_preds_score >= 0.5)) | ((matcha_O3_preds_score >= 0.5) & (h2p_O3_preds_score < 0.5))]

np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/sgmci_nodes.npy', sgmci_predicted_MCI_O3)
np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/sgmci_scores.npy', sgmci_predicted_MCI_O3_score)

np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/matcha_nodes.npy', matcha_predicted_MCI_O3)
np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/matcha_scores.npy', matcha_predicted_MCI_O3_score)

In [3]:
### 保留新预测的染色体内O3_MCI需要满足以下条件: 1)Repliseq>0; 2)属于相同的compartment; 3)至少有一个有SuperEnhancer

Repliseq_file = '/data/xujs/Public_Data/Repli-seq/hg38/GM12878/4DN/GM12878.Repli-seq.E2L_log2.1Mb.hg38.bedGraph'
Subcompartment_file = '/data/xujs/Project/HiC2PoreC/code/H2P/Analysis_Results/2024-05-06_embedding_visualization/liftOver_hg19_to_hg38/GSE63525_GM12878_subcompartments.hg19_to_hg38.1Mb.bed'
SuperEnhancer_file = '/data/xujs/References/human/hg38/SuperEnhancer/GM12878.SE.hg38_1Mb.bed'

# bedtools intersect -a ../hg38_windows/hg38.window_1Mb.bed -b GM12878.SE.hg38.bed -wa -wb | awk '{print $1,$2,$3}' >GM12878.SE.hg38_1Mb.bed

In [4]:
df_Repliseq = pd.read_table(Repliseq_file, sep='\t', header=None, names=['chrom', 'start', 'end', 'score'])
df_Repliseq = df_Repliseq.loc[~(df_Repliseq.end - df_Repliseq.start > 1000000)]
df_Repliseq = df_Repliseq.loc[~df_Repliseq.chrom.isin(["chrX", "chrY", "chrM"])]

df_Subcompartment = pd.read_table(Subcompartment_file, sep='\t', header=None).iloc[:, :4]
df_Subcompartment.columns =['chrom', 'start', 'end', 'subcompartment_type']

df_SuperEnhancer = pd.read_table(SuperEnhancer_file, sep='\t', header=None).iloc[:, :3]
df_SuperEnhancer.columns = ['chrom', 'start', 'end']
df_SuperEnhancer = df_SuperEnhancer.drop_duplicates()
df_SuperEnhancer['SE_type'] = 1
df_SuperEnhancer = df_SuperEnhancer.loc[~df_SuperEnhancer.chrom.isin(["chrX", "chrY", "chrM"])]

## 1Mb window
df_hg38_1Mb = pd.read_table('/data/xujs/References/human/hg38/hg38_windows/hg38.window_1Mb.bed', header=None)
df_hg38_1Mb.columns = ['chrom', 'start', 'end']
df_hg38_1Mb = df_hg38_1Mb.loc[~df_hg38_1Mb.chrom.isin(["chrX", "chrY", "chrM"])]

In [5]:
subcompartment_dict = {}
superenhancer_dict = {}
repliseq_dict = {}

for index, row in df_hg38_1Mb.iterrows():
    bin = f'{row["chrom"]}:{row["start"]}'
    # node = bin2node[bin]
    subcompartment_dict[bin] = 0
    superenhancer_dict[bin] = 0
    repliseq_dict[bin] = 0

for index, row in df_Subcompartment.iterrows():
    bin = f'{row["chrom"]}:{row["start"]}'
    subcompartment_dict[bin] = row['subcompartment_type']

for index, row in df_SuperEnhancer.iterrows():
    bin = f'{row["chrom"]}:{row["start"]}'
    superenhancer_dict[bin] = row['SE_type']

for index, row in df_Repliseq.iterrows():
    bin = f'{row["chrom"]}:{row["start"]}'
    repliseq_dict[bin] = row['score']

In [78]:
df_Subcompartment.to_csv('./GM12878_1Mb_SGMCI_MATCHA_comparison/GM12878_1Mb_subcompartment.tsv', sep='\t', header=True, index=False)

In [7]:
node2bin = np.load('/data/xujs/Project/DeepLearning/MCIP/Results/preprocess_results/hg38.1000kb.node2bin.npy', allow_pickle=True).item()
node2bin = {k-1:v for k,v in node2bin.items()}
bin2node = {v:k for k,v in node2bin.items()}

node2chrom = np.load(f"/data/xujs/Project/DeepLearning/MCIP/Results/preprocess_results/hg38.1000kb.node2chrom.npy", allow_pickle=True).item()
node2chrom = {k-1:v for k,v in node2chrom.items()}

data_dir = '/data/xujs/Project/DeepLearning/MCIP/Results'
chrom_range_ = np.load(f"{data_dir}/preprocess_results/hg38.1000kb.chrom_range.npy")
chrom_range = {}
for x in chrom_range_:
    chrom_range[x[0]] = np.array([int(x[1])-1, int(x[2])-1])

node_num = len(node2bin)
np.save('./GM12878_1Mb_SGMCI_MATCHA_comparison/hg38.1Mb.node2bin.npy', node2bin)

In [8]:
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

from matplotlib_venn import venn2

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from itertools import combinations
from scipy.stats import (
    mannwhitneyu,
    kruskal
)
from statsmodels.stats.multitest import multipletests

# ============================================================
# 1. Configuration
# ============================================================

# Input files
SGMCI_NODE_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/sgmci_nodes.npy"
SGMCI_SCORE_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/sgmci_scores.npy"

MATCHA_NODE_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/matcha_nodes.npy"
MATCHA_SCORE_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/matcha_scores.npy"

NODE2BIN_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/hg38.1Mb.node2bin.npy"
SUBCOMP_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/GM12878_1Mb_subcompartment.tsv"

# Optional bin-level annotation file
FEATURE_FILE = "./GM12878_1Mb_SGMCI_MATCHA_comparison/GM12878_1Mb_feature_tables/GM12878_1Mb_bin_features_log1p_zscore.tsv"

# Output directory
OUTDIR = Path("GM12878_1Mb_SGMCI_MATCHA_comparison")
OUTDIR.mkdir(exist_ok=True, parents=True)

# Analyze one MCI order at a time.
# Recommended: first analyze 3-way MCIs, then rerun for 4-way and 5-way.
MCI_ORDER = 3

# Because SGMCI and MATCHA scores are not necessarily calibrated
# on the same scale, top-K comparison is recommended.
SELECTION_MODE = "threshold"   ## optional: "topk"

# Set this according to your dataset size.
# If None, use the same number of predictions from both methods.
TOP_K = 100000

# Used only if SELECTION_MODE == "threshold"
SGMCI_THRESHOLD = 0.5
MATCHA_THRESHOLD = 0.5

BIN_SIZE = 1_000_000

GROUP_ORDER = [
    "SGMCI-specific",
    "MATCHA-specific"
]

BIO_FEATURES = [
    "ATAC-seq",
    "H3K27ac",
    "H3K4me1",
    "H3K4me3",
    "H3K27me3",
    "CTCF",
    "SMC3",
    "gene_count",
    "super_enhancer_overlap_bp"
]

# For gene number and super-enhancer overlap,
# total signal across all anchors is more intuitive.
TOTAL_FEATURES = {
    "gene_count",
    "super_enhancer_overlap_bp",
}


In [9]:
# ============================================================
# 2. General helper functions
# ============================================================

def parse_region(region, bin_size=1_000_000):
    """
    Convert:
        chr1:1000000
        chr1:1000000-2000000

    into:
        chrom, start, end
    """

    region = str(region)

    match = re.match(
        r"^(chr[^:]+):(\d+)(?:-(\d+))?$",
        region
    )

    if match is None:
        raise ValueError(
            f"Cannot parse genomic region: {region}"
        )

    chrom = match.group(1)
    start = int(match.group(2))

    if match.group(3) is None:
        end = start + bin_size
    else:
        end = int(match.group(3))

    return chrom, start, end


def canonical_candidate(nodes):
    """
    MCI is an unordered set of nodes.
    Sort node IDs before comparing prediction sets.
    """

    nodes = np.asarray(nodes).astype(int).ravel()

    # Remove padding values
    nodes = nodes[nodes >= 0]

    # Remove duplicated node IDs
    nodes = np.unique(nodes)

    return tuple(sorted(nodes.tolist()))


def safe_mean(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0 or np.all(np.isnan(values)):
        return np.nan

    return float(np.nanmean(values))


def safe_sum(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0 or np.all(np.isnan(values)):
        return np.nan

    return float(np.nansum(values))


def sample_for_plot(df, group_col, max_per_group=5000, seed=2026):
    """
    Subsample only for visualization.
    Statistical summaries remain based on all candidates.
    """

    if len(df) == 0:
        return df.copy()

    return (
        df.groupby(group_col, group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(len(x), max_per_group),
                random_state=seed
            )
        )
        .reset_index(drop=True)
    )


In [10]:
# ============================================================
# 3. Load node2bin and build 1-Mb annotation table
# ============================================================

node2bin = np.load(
    NODE2BIN_FILE,
    allow_pickle=True
).item()


bin_records = []

for node_id, region in node2bin.items():

    chrom, start, end = parse_region(
        region,
        bin_size=BIN_SIZE
    )

    bin_records.append({
        "node_id": int(node_id),
        "chrom": chrom,
        "start": start,
        "end": end,
        "center": (start + end) / 2
    })


bin_df = pd.DataFrame(bin_records)

bin_df = bin_df.sort_values(
    ["chrom", "start"]
).reset_index(drop=True)


# ============================================================
# 4. Add subcompartment and A/B compartment
# ============================================================

subcomp_df = pd.read_csv(
    SUBCOMP_FILE,
    sep="\t"
)

# Support several possible column names
subcomp_df = subcomp_df.rename(
    columns={
        "chromosome": "chrom",
        "subcompartment": "subcompartment_type"
    }
)

required_subcomp_cols = {
    "chrom",
    "start",
    "end",
    "subcompartment_type"
}

missing_subcomp = (
    required_subcomp_cols
    - set(subcomp_df.columns)
)

if missing_subcomp:
    raise ValueError(
        f"Missing subcompartment columns: {missing_subcomp}"
    )

subcomp_df["chrom"] = subcomp_df["chrom"].astype(str)
subcomp_df["start"] = subcomp_df["start"].astype(int)
subcomp_df["end"] = subcomp_df["end"].astype(int)

subcomp_map = (
    subcomp_df
    .drop_duplicates(["chrom", "start"])
    .set_index(["chrom", "start"])
    ["subcompartment_type"]
    .to_dict()
)

bin_df["subcompartment_type"] = [
    subcomp_map.get(
        (chrom, start),
        "X"
    )
    for chrom, start in zip(
        bin_df["chrom"],
        bin_df["start"]
    )
]

# A1/A2/... -> A
# B1/B2/... -> B
# X/unknown -> X
bin_df["compartment"] = (
    bin_df["subcompartment_type"]
    .astype(str)
    .str.extract(
        r"^([AB])",
        expand=False
    )
    .fillna("X")
)


In [11]:
# ============================================================
# 5. Add epigenomic, gene, and super-enhancer features
# ============================================================

if FEATURE_FILE is not None and Path(FEATURE_FILE).exists():

    feature_df = pd.read_csv(
        FEATURE_FILE,
        sep="\t"
    )

    feature_df = feature_df.rename(
        columns={
            "chromosome": "chrom"
        }
    )

    if "node_id" in feature_df.columns:

        feature_df["node_id"] = (
            feature_df["node_id"]
            .astype(int)
        )

        keep_cols = [
            "node_id"
        ] + [
            c for c in BIO_FEATURES
            if c in feature_df.columns
        ]

        bin_df = bin_df.merge(
            feature_df[keep_cols],
            on="node_id",
            how="left"
        )

    else:

        required_feature_keys = {
            "chrom",
            "start"
        }

        if not required_feature_keys.issubset(
            feature_df.columns
        ):
            raise ValueError(
                "Feature file must contain either "
                "node_id or chrom/start columns."
            )

        keep_cols = [
            "chrom",
            "start"
        ] + [
            c for c in BIO_FEATURES
            if c in feature_df.columns
        ]

        bin_df = bin_df.merge(
            feature_df[keep_cols],
            on=["chrom", "start"],
            how="left"
        )

else:

    print(
        "Feature file not found. "
        "Structural analyses will still run, "
        "but epigenomic analyses will be skipped."
    )


available_bio_features = [
    c for c in BIO_FEATURES
    if c in bin_df.columns
]

for col in available_bio_features:
    bin_df[col] = pd.to_numeric(
        bin_df[col],
        errors="coerce"
    )


annotation_by_node = (
    bin_df
    .set_index("node_id")
    .to_dict("index")
)


In [12]:
# ============================================================
# 6. Load prediction results
# ============================================================

def load_prediction_table(
    node_file,
    score_file,
    model_name,
    target_order=None
):
    """
    Load node arrays and score arrays.

    Supports:
        shape = (N, k)
        padded arrays with -1
        object arrays containing arrays/lists
    """

    raw_nodes = np.load(
        node_file,
        allow_pickle=True
    )

    raw_scores = np.load(
        score_file,
        allow_pickle=True
    ).reshape(-1)

    if len(raw_nodes) != len(raw_scores):
        raise ValueError(
            f"{model_name}: nodes and scores have different lengths."
        )

    records = []

    for nodes, score in zip(
        raw_nodes,
        raw_scores
    ):

        candidate = canonical_candidate(nodes)

        if target_order is not None:
            if len(candidate) != target_order:
                continue

        if len(candidate) < 2:
            continue

        if not np.isfinite(float(score)):
            continue

        records.append({
            "triplet": candidate,
            f"{model_name}_score": float(score)
        })

    df = pd.DataFrame(records)

    if len(df) == 0:
        raise ValueError(
            f"No valid {model_name} candidates found."
        )

    # If duplicates exist, retain the highest score.
    score_col = f"{model_name}_score"

    df = (
        df.sort_values(
            score_col,
            ascending=False
        )
        .drop_duplicates(
            subset=["triplet"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    return df


sgmci_df = load_prediction_table(
    SGMCI_NODE_FILE,
    SGMCI_SCORE_FILE,
    model_name="sgmci",
    target_order=MCI_ORDER
)

matcha_df = load_prediction_table(
    MATCHA_NODE_FILE,
    MATCHA_SCORE_FILE,
    model_name="matcha",
    target_order=MCI_ORDER
)


In [13]:
# ============================================================
# 7. Select equal-sized prediction sets
# ============================================================

def select_predictions(
    df,
    model_name,
    mode="topk",
    top_k=None,
    threshold=None
):

    score_col = f"{model_name}_score"

    if mode == "topk":

        if top_k is None:
            top_k = len(df)

        top_k = min(top_k, len(df))

        return (
            df.sort_values(
                score_col,
                ascending=False
            )
            .head(top_k)
            .copy()
        )

    if mode == "threshold":

        if threshold is None:
            raise ValueError(
                f"Threshold missing for {model_name}."
            )

        return df[
            df[score_col] >= threshold
        ].copy()

    raise ValueError(
        "mode must be 'topk' or 'threshold'."
    )


if SELECTION_MODE == "topk":

    if TOP_K is None:
        TOP_K = min(
            len(sgmci_df),
            len(matcha_df)
        )

    sgmci_selected = select_predictions(
        sgmci_df,
        model_name="sgmci",
        mode="topk",
        top_k=TOP_K
    )

    matcha_selected = select_predictions(
        matcha_df,
        model_name="matcha",
        mode="topk",
        top_k=TOP_K
    )

else:

    sgmci_selected = select_predictions(
        sgmci_df,
        model_name="sgmci",
        mode="threshold",
        threshold=SGMCI_THRESHOLD
    )

    matcha_selected = select_predictions(
        matcha_df,
        model_name="matcha",
        mode="threshold",
        threshold=MATCHA_THRESHOLD
    )

In [16]:

# ============================================================
# 8. Compare shared and method-specific sets
# ============================================================

sgmci_set = set(
    sgmci_selected["triplet"]
)

matcha_set = set(
    matcha_selected["triplet"]
)

shared_set = sgmci_set & matcha_set
sgmci_only_set = sgmci_set - matcha_set
matcha_only_set = matcha_set - sgmci_set

comparison = pd.merge(
    sgmci_selected,
    matcha_selected,
    on="triplet",
    how="outer"
)

comparison["prediction_group"] = np.select(
    [
        comparison["sgmci_score"].notna()
        & comparison["matcha_score"].notna(),

        comparison["sgmci_score"].notna()
        & comparison["matcha_score"].isna(),

        comparison["sgmci_score"].isna()
        & comparison["matcha_score"].notna()
    ],
    [
        "Shared",
        "SGMCI-specific",
        "MATCHA-specific"
    ],
    default="Unknown"
)

comparison["triplet_id"] = comparison[
    "triplet"
].apply(
    lambda x: "_".join(
        map(str, x)
    )
)

comparison.to_csv(
    OUTDIR / "prediction_set_comparison.tsv",
    sep="\t",
    index=False
)

# print("SGMCI predictions:", len(sgmci_set))
# print("MATCHA predictions:", len(matcha_set))
print("Shared predictions:", len(shared_set))
# print("SGMCI-specific:", len(sgmci_only_set))
# print("MATCHA-specific:", len(matcha_only_set))

jaccard_index = len(shared_set) / len(
    sgmci_set | matcha_set
)

overlap_coefficient = len(shared_set) / min(
    len(sgmci_set),
    len(matcha_set)
)

print("Jaccard index:", round(jaccard_index, 4))
print(
    "Overlap coefficient:",
    round(overlap_coefficient, 4)
)


Shared predictions: 105966
Jaccard index: 0.2725
Overlap coefficient: 0.8611


In [17]:
# ============================================================
# 9. Candidate-level genomic feature extraction
# ============================================================

def extract_candidate_features(
    candidate,
    annotation_by_node,
    available_bio_features
):

    records = []

    for node_id in candidate:

        node_id = int(node_id)

        if node_id not in annotation_by_node:
            return {
                "triplet": candidate
            }

        records.append(
            annotation_by_node[node_id]
        )

    ann = pd.DataFrame(records)

    chroms = ann["chrom"].tolist()
    centers = ann["center"].astype(float).tolist()

    # Sort anchors by chromosome and genomic position
    order = np.lexsort(
        (
            ann["start"].to_numpy(),
            ann["chrom"].astype(str).to_numpy()
        )
    )

    ann_ordered = ann.iloc[order].reset_index(
        drop=True
    )

    chroms_ordered = ann_ordered["chrom"].tolist()
    centers_ordered = (
        ann_ordered["center"]
        .astype(float)
        .tolist()
    )

    is_cis = len(set(chroms)) == 1

    pairwise_distance_mb = []

    if is_cis:

        for i, j in combinations(
            range(len(centers_ordered)),
            2
        ):

            distance_mb = abs(
                centers_ordered[i]
                - centers_ordered[j]
            ) / 1e6

            pairwise_distance_mb.append(
                distance_mb
            )

        sorted_centers = sorted(
            centers_ordered
        )

        adjacent_spacing_mb = [
            abs(
                sorted_centers[i + 1]
                - sorted_centers[i]
            ) / 1e6
            for i in range(
                len(sorted_centers) - 1
            )
        ]

        min_pairwise_distance_mb = min(
            pairwise_distance_mb
        )

        mean_pairwise_distance_mb = np.mean(
            pairwise_distance_mb
        )

        max_pairwise_distance_mb = max(
            pairwise_distance_mb
        )

        total_span_mb = (
            max(centers_ordered)
            - min(centers_ordered)
        ) / 1e6

        min_anchor_spacing_mb = min(
            adjacent_spacing_mb
        )

        mean_anchor_spacing_mb = np.mean(
            adjacent_spacing_mb
        )

    else:

        min_pairwise_distance_mb = np.nan
        mean_pairwise_distance_mb = np.nan
        max_pairwise_distance_mb = np.nan
        total_span_mb = np.nan
        min_anchor_spacing_mb = np.nan
        mean_anchor_spacing_mb = np.nan

    compartments = (
        ann_ordered["compartment"]
        .astype(str)
        .tolist()
    )

    subcompartments = (
        ann_ordered["subcompartment_type"]
        .astype(str)
        .tolist()
    )

    result = {
        "triplet": candidate,
        "interaction_type": (
            "cis" if is_cis else "trans"
        ),
        "n_anchors": len(candidate),
        "n_chromosomes": len(set(chroms)),
        "min_pairwise_distance_mb":
            min_pairwise_distance_mb,
        "mean_pairwise_distance_mb":
            mean_pairwise_distance_mb,
        "max_pairwise_distance_mb":
            max_pairwise_distance_mb,
        "total_span_mb":
            total_span_mb,
        "min_anchor_spacing_mb":
            min_anchor_spacing_mb,
        "mean_anchor_spacing_mb":
            mean_anchor_spacing_mb,
        "compartment_pattern": "|".join(
            compartments
        ),
        "subcompartment_pattern": "|".join(
            subcompartments
        ),
        "n_A_anchors": compartments.count("A"),
        "n_B_anchors": compartments.count("B"),
        "n_X_anchors": compartments.count("X"),
        "n_unique_compartments": len(
            set(compartments)
        ),
        "n_unique_subcompartments": len(
            set(subcompartments)
        ),
        "all_same_compartment": (
            len(set(compartments)) == 1
        ),
        "all_same_subcompartment": (
            len(set(subcompartments)) == 1
        )
    }

    # Candidate-level summaries of biological signals
    for feature in available_bio_features:

        values = (
            ann_ordered[feature]
            .astype(float)
            .to_numpy()
        )

        result[f"{feature}_mean"] = safe_mean(
            values
        )

        result[f"{feature}_total"] = safe_sum(
            values
        )

        result[f"{feature}_max"] = (
            np.nanmax(values)
            if not np.all(np.isnan(values))
            else np.nan
        )

    return result


candidate_feature_records = []

for candidate in comparison["triplet"]:

    candidate_feature_records.append(
        extract_candidate_features(
            candidate,
            annotation_by_node,
            available_bio_features
        )
    )

candidate_feature_df = pd.DataFrame(
    candidate_feature_records
)

comparison = comparison.merge(
    candidate_feature_df,
    on="triplet",
    how="left"
)

comparison.to_csv(
    OUTDIR / "prediction_comparison_with_features.tsv",
    sep="\t",
    index=False
)


In [18]:
# ============================================================
# 10. Candidate-level summary statistics
# ============================================================

summary_cols = [
    "interaction_type",
    "min_pairwise_distance_mb",
    "mean_pairwise_distance_mb",
    "max_pairwise_distance_mb",
    "total_span_mb",
    "min_anchor_spacing_mb",
    "mean_anchor_spacing_mb",
    "n_A_anchors",
    "n_B_anchors",
    "n_X_anchors",
    "n_unique_compartments",
    "n_unique_subcompartments",
]

summary_records = []

for group, group_df in comparison.groupby(
    "prediction_group",
    sort=False
):

    for col in summary_cols:

        if col == "interaction_type":
            continue

        values = pd.to_numeric(
            group_df[col],
            errors="coerce"
        ).dropna()

        if len(values) == 0:
            continue

        summary_records.append({
            "prediction_group": group,
            "feature": col,
            "N": len(values),
            "mean": values.mean(),
            "median": values.median(),
            "Q1": values.quantile(0.25),
            "Q3": values.quantile(0.75),
            "min": values.min(),
            "max": values.max()
        })

summary_df = pd.DataFrame(
    summary_records
)

summary_df.to_csv(
    OUTDIR / "candidate_feature_summary.tsv",
    sep="\t",
    index=False
)


In [19]:
# ============================================================
# 11. Statistical tests
# ============================================================

def calculate_pairwise_stats(
    df,
    value_col,
    group_col="prediction_group"
):

    groups = [
        g for g in GROUP_ORDER
        if g in df[group_col].unique()
    ]

    values_by_group = {
        group: pd.to_numeric(
            df.loc[
                df[group_col] == group,
                value_col
            ],
            errors="coerce"
        ).dropna().to_numpy()
        for group in groups
    }

    values_by_group = {
        group: values
        for group, values in values_by_group.items()
        if len(values) > 0
    }

    if len(values_by_group) < 2:
        return pd.DataFrame()

    kw_result = kruskal(
        *values_by_group.values()
    )

    pairwise_records = []

    for g1, g2 in combinations(
        values_by_group.keys(),
        2
    ):

        u_result = mannwhitneyu(
            values_by_group[g1],
            values_by_group[g2],
            alternative="two-sided"
        )

        pairwise_records.append({
            "value": value_col,
            "test": "Mann-Whitney U",
            "group_1": g1,
            "group_2": g2,
            "statistic": u_result.statistic,
            "p_value": u_result.pvalue,
            "kruskal_p_value": kw_result.pvalue
        })

    result = pd.DataFrame(
        pairwise_records
    )

    if len(result) > 0:
        result["BH_FDR"] = multipletests(
            result["p_value"],
            method="fdr_bh"
        )[1]

    return result


statistic_tables = []

stat_cols = [
    "min_pairwise_distance_mb",
    "mean_pairwise_distance_mb",
    "max_pairwise_distance_mb",
    "total_span_mb",
    "min_anchor_spacing_mb",
    "mean_anchor_spacing_mb",
    "n_A_anchors",
    "n_B_anchors",
    "n_unique_compartments",
    "n_unique_subcompartments",
]

for col in stat_cols:

    if col not in comparison.columns:
        continue

    result = calculate_pairwise_stats(
        comparison,
        value_col=col
    )

    if len(result) > 0:
        statistic_tables.append(result)

if len(statistic_tables) > 0:

    statistics_df = pd.concat(
        statistic_tables,
        ignore_index=True
    )

    statistics_df.to_csv(
        OUTDIR / "pairwise_statistics.tsv",
        sep="\t",
        index=False
    )


In [20]:
# ============================================================
# 12. Visualization: prediction-set overlap
# ============================================================

plt.figure(
    figsize=(6, 5),
    dpi=180
)

venn = venn2(
    subsets=[
        len(sgmci_only_set),
        len(matcha_only_set),
        len(shared_set)
    ],
    set_labels=[
        "SGMCI",
        "MATCHA"
    ]
)

plt.title(
    f"GM12878 {MCI_ORDER}-way MCI predictions"
)

plt.tight_layout()

plt.savefig(
    OUTDIR / "prediction_set_venn.pdf",
    bbox_inches="tight"
)

plt.savefig(
    OUTDIR / "prediction_set_venn.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial


In [21]:
# ============================================================
# 13. Visualization: prediction group sizes
# ============================================================

group_size_df = (
    comparison["prediction_group"]
    .value_counts()
    .reindex(
        GROUP_ORDER,
        fill_value=0
    )
    .rename("N")
    .reset_index()
    .rename(
        columns={
            "index": "prediction_group"
        }
    )
)

plt.figure(
    figsize=(7, 5),
    dpi=180
)

sns.barplot(
    data=group_size_df,
    x="prediction_group",
    y="N",
    order=GROUP_ORDER,
    color="#756BB1"
)

plt.xlabel("")
plt.ylabel("Number of predicted MCIs")
plt.title(
    "Prediction-set sizes"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    OUTDIR / "prediction_group_sizes.pdf",
    bbox_inches="tight"
)

plt.close()


In [15]:
# # ============================================================
# # 14. Visualization: cis/trans distribution
# # ============================================================

# cis_trans = pd.crosstab(
#     comparison["prediction_group"],
#     comparison["interaction_type"]
# )

# cis_trans = cis_trans.reindex(
#     GROUP_ORDER,
#     fill_value=0
# )

# cis_trans_fraction = (
#     cis_trans.div(
#         cis_trans.sum(axis=1),
#         axis=0
#     )
#     .fillna(0)
# )

# ax = cis_trans_fraction[
#     ["cis", "trans"]
# ].plot(
#     kind="bar",
#     stacked=True,
#     figsize=(7, 5),
#     color=[
#         "#3182BD",
#         "#E6550D"
#     ],
#     width=0.75
# )

# ax.set_xlabel("")
# ax.set_ylabel("Fraction of predicted MCIs")
# ax.set_title(
#     "Cis/trans composition"
# )

# ax.set_xticklabels(
#     ax.get_xticklabels(),
#     rotation=20,
#     ha="right"
# )

# ax.legend(
#     title="Interaction type",
#     frameon=False
# )

# plt.tight_layout()

# plt.savefig(
#     OUTDIR / "cis_trans_distribution.pdf",
#     bbox_inches="tight"
# )

# plt.close()



KeyError: "['trans'] not in index"

In [22]:
# ============================================================
# 15. Visualization: genomic distance and span
# ============================================================

distance_cols = [
    "min_pairwise_distance_mb",
    "mean_pairwise_distance_mb",
    "max_pairwise_distance_mb",
    "total_span_mb",
    "min_anchor_spacing_mb",
    "mean_anchor_spacing_mb",
]

distance_labels = {
    "min_pairwise_distance_mb":
        "Minimum pairwise distance (Mb)",
    "mean_pairwise_distance_mb":
        "Mean pairwise distance (Mb)",
    "max_pairwise_distance_mb":
        "Maximum pairwise distance (Mb)",
    "total_span_mb":
        "Total genomic span (Mb)",
    "min_anchor_spacing_mb":
        "Minimum adjacent-anchor spacing (Mb)",
    "mean_anchor_spacing_mb":
        "Mean adjacent-anchor spacing (Mb)"
}

cis_only = comparison[
    comparison["interaction_type"] == "cis"
].copy()

distance_plot_df = cis_only.melt(
    id_vars=["prediction_group"],
    value_vars=distance_cols,
    var_name="feature",
    value_name="value"
).dropna()

distance_plot_df["feature_label"] = (
    distance_plot_df["feature"]
    .map(distance_labels)
)

distance_plot_df = sample_for_plot(
    distance_plot_df,
    group_col="prediction_group",
    max_per_group=5000
)

g = sns.catplot(
    data=distance_plot_df,
    x="prediction_group",
    y="value",
    col="feature_label",
    col_wrap=3,
    kind="box",
    order=GROUP_ORDER,
    showfliers=False,
    color="#B3CDE3",
    height=4,
    aspect=1.2,
    sharey=False
)

g.set_titles("{col_name}")
g.set_axis_labels("", "")
g.set_xticklabels(rotation=25, ha="right")

plt.tight_layout()

g.savefig(
    OUTDIR / "genomic_distance_distributions.pdf"
)

plt.close()


In [23]:
# ============================================================
# 16. Visualization: compartment composition
# ============================================================

compartment_fraction = pd.crosstab(
    comparison["prediction_group"],
    comparison["compartment_pattern"],
    normalize="index"
).reindex(
    GROUP_ORDER,
    fill_value=0
)

plt.figure(
    figsize=(18, 3),
    dpi=180
)

sns.heatmap(
    compartment_fraction,
    cmap="Blues",
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    cbar_kws={
        "label": "Fraction of candidates"
    }
)

plt.xlabel("Compartment pattern")
plt.ylabel("")
plt.title(
    "Compartment-pattern composition"
)

plt.tight_layout()

plt.savefig(
    OUTDIR / "compartment_pattern_heatmap.pdf",
    bbox_inches="tight"
)

plt.close()

In [24]:
# ============================================================
# 17. Anchor-level compartment and subcompartment composition
# ============================================================

anchor_records = []

for row in comparison.itertuples(
    index=False
):

    candidate = row.triplet
    group = row.prediction_group

    for position, node_id in enumerate(
        candidate,
        start=1
    ):

        node_id = int(node_id)

        if node_id not in annotation_by_node:
            continue

        ann = annotation_by_node[node_id]

        anchor_records.append({
            "triplet": candidate,
            "prediction_group": group,
            "anchor_position": position,
            "node_id": node_id,
            "chrom": ann["chrom"],
            "start": ann["start"],
            "compartment": ann[
                "compartment"
            ],
            "subcompartment_type": ann[
                "subcompartment_type"
            ]
        })

anchor_df = pd.DataFrame(
    anchor_records
)

anchor_df.to_csv(
    OUTDIR / "anchor_level_annotations.tsv",
    sep="\t",
    index=False
)

anchor_compartment_fraction = pd.crosstab(
    anchor_df["prediction_group"],
    anchor_df["compartment"],
    normalize="index"
).reindex(
    GROUP_ORDER,
    fill_value=0
)

plt.figure(
    figsize=(7, 5),
    dpi=180
)

anchor_compartment_fraction[
    ["A", "B", "X"]
].plot(
    kind="bar",
    stacked=False,
    color=[
        "#E41A1C",
        "#377EB8",
        "#BDBDBD"
    ],
    width=0.75
)

plt.xlabel("")
plt.ylabel("Fraction of anchors")
plt.title(
    "A/B compartment composition of MCI anchors"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.legend(
    title="Compartment",
    frameon=False
)

plt.tight_layout()

plt.savefig(
    OUTDIR / "anchor_compartment_composition.pdf",
    bbox_inches="tight"
)

plt.close()


# Keep the most frequent subcompartments visible
top_subcomp = (
    anchor_df["subcompartment_type"]
    .value_counts()
    .head(15)
    .index
)

anchor_df["subcompartment_plot"] = (
    anchor_df["subcompartment_type"]
    .where(
        anchor_df["subcompartment_type"]
        .isin(top_subcomp),
        "Other"
    )
)

subcomp_fraction = pd.crosstab(
    anchor_df["prediction_group"],
    anchor_df["subcompartment_plot"],
    normalize="index"
).reindex(
    GROUP_ORDER,
    fill_value=0
)

subcomp_fraction.T.plot(
    kind="bar",
    stacked=False,
    figsize=(12, 5),
    colormap="tab20"
)

plt.xlabel("Subcompartment")
plt.ylabel("Fraction of anchors")
plt.title(
    "Subcompartment composition of MCI anchors"
)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.legend(
    title="Prediction group",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()

plt.savefig(
    OUTDIR / "anchor_subcompartment_composition.pdf",
    bbox_inches="tight"
)

plt.close()

<Figure size 1260x900 with 0 Axes>

In [25]:
# ============================================================
# 18. Visualization: epigenomic and biological features
# ============================================================

plot_feature_map = {}

for feature in available_bio_features:

    if feature in TOTAL_FEATURES:
        plot_feature_map[
            feature
        ] = f"{feature}_total"
    else:
        plot_feature_map[
            feature
        ] = f"{feature}_mean"


bio_plot_cols = [
    col for col in plot_feature_map.values()
    if col in comparison.columns
]

if len(bio_plot_cols) > 0:

    bio_long_records = []

    for feature, value_col in plot_feature_map.items():

        if value_col not in comparison.columns:
            continue

        tmp = comparison[
            [
                "prediction_group",
                value_col
            ]
        ].copy()

        tmp = tmp.rename(
            columns={
                value_col: "value"
            }
        )

        tmp["feature"] = feature

        bio_long_records.append(
            tmp
        )

    bio_long_df = pd.concat(
        bio_long_records,
        ignore_index=True
    ).dropna()

    bio_long_df = sample_for_plot(
        bio_long_df,
        group_col="prediction_group",
        max_per_group=5000
    )


    g = sns.catplot(
        data=bio_long_df,
        x="prediction_group",
        y="value",
        col="feature",
        col_wrap=3,
        kind="box",
        order=GROUP_ORDER,
        showfliers=False,
        color="#B3CDE3",
        height=4,
        aspect=1.25,
        sharey=False
    )

    g.set_titles("{col_name}")
    g.set_axis_labels("", "")
    g.set_xticklabels(
        rotation=25,
        ha="right"
    )

    plt.tight_layout()

    g.savefig(
        OUTDIR / "biological_feature_distributions.pdf"
    )

    plt.close()


    # Feature-level standardized group means
    group_means = (
        comparison
        .groupby("prediction_group")[
            bio_plot_cols
        ]
        .mean()
        .reindex(GROUP_ORDER)
    )

    feature_names = [
        feature
        for feature, value_col
        in plot_feature_map.items()
        if value_col in group_means.columns
    ]

    group_means.columns = feature_names

    feature_z = (
        group_means
        - group_means.mean(axis=0)
    ) / group_means.std(
        axis=0,
        ddof=0
    ).replace(0, np.nan)

    plt.figure(
        figsize=(10, 5),
        dpi=180
    )

    sns.heatmap(
        feature_z.T,
        cmap="RdBu_r",
        center=0,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={
            "label": "Z-score of group mean"
        }
    )

    plt.xlabel("")
    plt.ylabel("")
    plt.title(
        "Relative biological-feature differences"
    )

    plt.tight_layout()

    plt.savefig(
        OUTDIR / "biological_feature_group_mean_heatmap.pdf",
        bbox_inches="tight"
    )

    plt.close()


print(
    f"Analysis completed. Results saved to: {OUTDIR}"
)

Analysis completed. Results saved to: GM12878_1Mb_SGMCI_MATCHA_comparison
